In [147]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [148]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [200]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria" : "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages=[]
    add_user_message(messages=messages,text=prompt)
    add_assistant_message(messages=messages,text="```json")
    response=chat(messages=messages,stop_sequences=["```"])
    return json.loads(response)

In [201]:
dataset = generate_dataset()

dataset 

[{'task': 'Create a JSON configuration object for an AWS Lambda function that processes S3 bucket events, with environment variables for bucket name and log level',
  'format': 'json',
  'solution_criteria': 'Must include valid Lambda configuration with handler, runtime, environment variables section containing bucket name and log level, and proper JSON syntax'},
 {'task': 'Write a Python function that takes an AWS S3 object key and returns the file extension. Handle keys with multiple dots and keys without extensions',
  'format': 'python',
  'solution_criteria': "Function must correctly extract file extensions from S3 keys (e.g., 'folder/image.tar.gz' returns 'gz'), handle edge cases like no extension, and be reusable"},
 {'task': 'Write a regex pattern that matches valid AWS IAM role ARNs in the format: arn:aws:iam::account-id:role/role-name, where role-name may contain hyphens and underscores',
  'format': 'regex',
  'solution_criteria': 'Pattern must validate the ARN structure, en

In [202]:
with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

In [190]:
def run_prompt(test_case):
    prompt=f"""
    please solve the following task : 
    
    {test_case["task"]}
    
    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """
    messages=[]
    add_user_message(messages=messages,text=prompt)
    add_assistant_message(messages=messages,text="```code")
    response=chat(messages=messages,stop_sequences=["```"])
    return response

In [210]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>


Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    
    messages=[]
    add_user_message(messages=messages,text=eval_prompt)
    add_assistant_message(messages=messages,text="```json")
    response=chat(messages=messages,stop_sequences=["```"])
    return json.loads(response)

In [204]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [211]:
def run_test_case(test_case):
    
    response=run_prompt(test_case=test_case)
    
    result = grade_by_model(test_case=test_case,output=response)
    model_grade= result["score"]
    reasoning=result["reasoning"]
    
    syntax_grade=grade_syntax(response=response,test_case=test_case)
    grade = (model_grade+syntax_grade)/2
    
    print("test case : ",test_case)
    print("model grade : " , model_grade)
    print("syntax grade : " , syntax_grade)
    print("grade : " , grade)
    
    return{
        "test_case":test_case,
        "response":response,
        "grade":grade,
        "reasoning":reasoning
    }

In [212]:
from statistics import mean
def run_evals(dataset):
    results=[]
    for test_case in dataset:
        result = run_test_case(test_case=test_case)
        results.append(result)
    average_score= mean([result["grade"] for result in results])
    print(f"Average Score : {average_score}")
    return results

In [213]:
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_evals(dataset=dataset)

test case :  {'task': 'Create a JSON configuration object for an AWS Lambda function that processes S3 bucket events, with environment variables for bucket name and log level', 'format': 'json', 'solution_criteria': 'Must include valid Lambda configuration with handler, runtime, environment variables section containing bucket name and log level, and proper JSON syntax'}
model grade :  5
syntax grade :  10
grade :  7.5
test case :  {'task': 'Write a Python function that takes an AWS S3 object key and returns the file extension. Handle keys with multiple dots and keys without extensions', 'format': 'python', 'solution_criteria': "Function must correctly extract file extensions from S3 keys (e.g., 'folder/image.tar.gz' returns 'gz'), handle edge cases like no extension, and be reusable"}
model grade :  7
syntax grade :  10
grade :  8.5
test case :  {'task': 'Write a regex pattern that matches valid AWS IAM role ARNs in the format: arn:aws:iam::account-id:role/role-name, where role-name ma

In [214]:
print(json.dumps(results,indent=2))

[
  {
    "test_case": {
      "task": "Create a JSON configuration object for an AWS Lambda function that processes S3 bucket events, with environment variables for bucket name and log level",
      "format": "json",
      "solution_criteria": "Must include valid Lambda configuration with handler, runtime, environment variables section containing bucket name and log level, and proper JSON syntax"
    },
    "response": "\n{\n  \"FunctionName\": \"S3EventProcessor\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-execution-role\",\n  \"Handler\": \"index.handler\",\n  \"Timeout\": 60,\n  \"MemorySize\": 256,\n  \"Environment\": {\n    \"Variables\": {\n      \"BUCKET_NAME\": \"my-s3-bucket\",\n      \"LOG_LEVEL\": \"INFO\"\n    }\n  },\n  \"Events\": {\n    \"S3Event\": {\n      \"Type\": \"S3\",\n      \"Properties\": {\n        \"Bucket\": \"my-s3-bucket\",\n        \"Events\": [\n          \"s3:ObjectCreated:*\",\n          \"s3:ObjectRemoved:*\"\